# Method 4 — RAG with Hybrid Augmentation

RAG Dynamic with eight targeted fixes addressing specific failure modes identified during error analysis on a dev split (3,999 rows). 
It uses the remaining 4,001 rows as a held-out test set.
    
Enhancements over RAG Dynamic:
1. FIX 1 : **Severity rubric:** A 4-point severity scale for severe_toxic (mild/moderate/severe/extreme) with explicit score ranges, plus a dedicated second-pass GPT call for borderline cases.
2. FIX 2 : **Threat precision:** Boundary examples distinguishing real threats from hyperbole ("I will find you and kill you" vs "fuck off and die"), with target-proximity regex filtering.
3. FIX 3 : **Identity hate recall:** Group-noun detection boosting and a dedicated second-pass prompt for borderline identity_hate cases.
4. FIX 4 : **Skip gate:** Ultra-clean comments bypass GPT entirely, saving cost and reducing false positives.
5. FIX 5 : **Heuristic rules** Post-prediction logic enforcing label co-occurrence constraints (e.g., severe_toxic → toxic).
6. FIX 7 : **Common mistake warnings:** The prompt explicitly warns about LLM failure modes: over-labelling anger as threat, missing coded hate speech, conflating intensity with severity.
7. FIX 8 : **Dev-tuned thresholds:** Per-label threshold optimisation with recall floors (severe_toxic ≥ 0.50, identity_hate ≥ 0.60) and fine granularity (step=2).
 
This method using `text-embedding-3-small`.Hard labels always include at least one extreme example; normal labels use nearest neighbours.
Confidence scores (0-100).

**Prerequisite:** `few_shot_pool.csv` must be present (the training/pool CSV).
On first run the pool embeddings are computed and cached to `pool_embeddings.npz`.


In [ ]:
# ============================================================
# Method 4 — RAG Multi-label Optimization
# ============================================================
#

import os, re, json, math, time, hashlib, asyncio
import numpy as np
import pandas as pd
import tiktoken

from openai import OpenAI, AsyncOpenAI
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score,
    hamming_loss, jaccard_score, multilabel_confusion_matrix
)

# ============================================================
# CONFIG
# ============================================================

# ── Files ─────────────────────────────────────────────────────

FEW_SHOT_CSV   = "/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/few_shot_pool_fixed_3.csv"
TEST_CSV       = "/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/stress_test_eval_set.csv"

DEV_CSV        = "eval_dev_cl1.csv"
HELD_TEST_CSV  = "eval_test_cl1.csv"

INDEX_FILE     = "label_aware_index_v5_cl1.npz"
META_FILE      = "label_aware_meta_v5_cl1.json"
CURATED_FILE   = "curated_hard_labels_cl1.json"

OUTPUT_DEV_CSV  = "predictions_v5_dev_cl1.csv"
OUTPUT_TEST_CSV = "predictions_v5_test_cl1.csv"
DEV_THRESHOLDS  = "threshold_from_dev_v5_cl1.json"

# ── Models ────────────────────────────────────────────────────
EMBED_MODEL    = "text-embedding-3-small"
GPT_MODEL      = "gpt-4.1"

# ── Batch / Retrieval ─────────────────────────────────────────
EMBED_BATCH       = 512
RETRIEVE_K        = 6      # FIX 3: raised from 5 for better hard-label coverage
RETRIEVE_K_HIGH   = 8      # FIX 3: raised from 7
ASYNC_CONCURRENCY = 8

# ── Labels ────────────────────────────────────────────────────
LABEL_COLS         = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]
HARD_LABELS        = ["severe_toxic","threat","identity_hate"]
SOFT_GUARD_LABELS  = ["insult", "obscene"]
SOFT_GUARD_FLOOR   = 0.04
PRECISION_PRIORITY = ["severe_toxic","threat","insult"]

# ── Per-label confidence thresholds (initial; overridden by dev tuning) ──
LABEL_THRESHOLDS = {
    "toxic":          25,
    "severe_toxic":   35,    # FIX 1: raised from 75 (v4 tuned to 5, indicating miscalibration)
    "obscene":        30,
    "threat":         45,    # FIX 2: raised — rely on better prompt rather than low threshold
    "insult":         45,
    "identity_hate":  35,    # FIX 3: lowered to catch more subtle cases
}

# ── Skip-gate (FIX 4: made more aggressive for genuinely clean comments) ──
HARD_LABEL_HINT_FLOOR  = 0.08
SKIP_CLEAN_RATIO       = 0.95    # FIX 4: relaxed from 0.97 for clean comments
SKIP_TOP1_SIM          = 0.90    # FIX 4: relaxed from 0.92
SKIP_ALL_HINT_FLOOR    = 0.05
SHORT_COMMENT_WORDS    = 12      # FIX 4: lowered from 15

# ── Self-consistency (FIX 6: adaptive) ────────────────────────
BORDERLINE_LOW  = 25
BORDERLINE_HIGH = 75
# Only these labels trigger self-consistency (FIX 6)
CONSISTENCY_LABELS = ["severe_toxic", "threat", "identity_hate"]
CONSISTENCY_WEIGHT_PASS1 = 0.6   # FIX 6: weighted merge
CONSISTENCY_WEIGHT_PASS2 = 0.4

# ── v5 index settings ─────────────────────────────────────────
LOCAL_CENTROID_K = 50
PURITY_FLOOR = {
    "severe_toxic":  0.45,
    "threat":        0.55,
    "identity_hate": 0.50,   # FIX 3: slightly lowered to include more diverse examples
}

# ── Toxic keywords for clean-centroid purification ────────────
TOXIC_KEYWORDS = [
    "fuck","shit","bitch","asshole","cunt","dick","pussy","cock",
    "nigger","faggot","retard","kill","die","rape","whore","bastard",
    "idiot","moron","stupid","hate","loser"
]

# FIX 3: Group-identity nouns that boost identity_hate hint
GROUP_IDENTITY_NOUNS = [
    "muslim", "muslims", "islam", "islamic", "jew", "jews", "jewish",
    "black", "blacks", "white", "whites", "asian", "asians", "chinese",
    "mexican", "mexicans", "immigrant", "immigrants", "refugee", "refugees",
    "gay", "gays", "lesbian", "lesbians", "trans", "transgender", "lgbtq",
    "women", "woman", "feminist", "feminists", "christian", "christians",
    "hindu", "hindus", "sikh", "sikhs", "arab", "arabs", "african",
    "hispanic", "latino", "latina", "native", "indigenous",
]

# ── Toggles ───────────────────────────────────────────────────
COT_ENABLED = False
APPLY_HEURISTIC_RULES = True

# ── Pricing ───────────────────────────────────────────────────
EMBED_COST_PER_1K      = 0.00002
GPT_INPUT_COST_PER_1K  = 0.00080
GPT_OUTPUT_COST_PER_1K = 0.00320

# ── Reproducibility ───────────────────────────────────────────
GPT_SEED = 42

# ── Clients ───────────────────────────────────────────────────
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Please set it before running.")

client       = OpenAI(api_key=api_key)
async_client = AsyncOpenAI(api_key=api_key)

try:
    _enc = tiktoken.encoding_for_model(GPT_MODEL)
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")


# ============================================================
# DEV / TEST SPLIT
# ============================================================

def split_eval_set(test_size: float = 0.5, random_state: int = 42):
    """
    Splits stress_test_eval_set.csv into stratified dev + held-out test.
    Rare combos (n=1) forced into test set.
    """
    print("=" * 72)
    print("SPLITTING EVAL SET (dev / held-out test)")
    print("=" * 72)

    df = pd.read_csv(TEST_CSV)
    df["combo"] = df[LABEL_COLS].apply(
        lambda r: "+".join(c for c in LABEL_COLS if r[c] == 1) or "clean", axis=1
    )

    rare_mask = df["combo"].map(df["combo"].value_counts()) == 1
    common_df = df[~rare_mask]
    rare_df   = df[rare_mask]

    dev_idx, test_idx = train_test_split(
        common_df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=common_df["combo"]
    )

    dev  = df.loc[dev_idx].drop(columns=["combo"])
    test = pd.concat([df.loc[test_idx], rare_df]).drop(columns=["combo"])

    dev.to_csv(DEV_CSV, index=False)
    test.to_csv(HELD_TEST_CSV, index=False)

    print(f"Dev  rows : {len(dev):,}  → {DEV_CSV}")
    print(f"Test rows : {len(test):,}  → {HELD_TEST_CSV}")
    print(f"\n  {'Label':<20} {'Full%':>7} {'Dev%':>7} {'Test%':>7}")
    for c in LABEL_COLS:
        fp = 100 * df[c].mean()
        dp = 100 * dev[c].mean()
        tp = 100 * test[c].mean()
        print(f"  {c:<20} {fp:>7.2f} {dp:>7.2f} {tp:>7.2f}")
    print("=" * 72)


# ============================================================
# UTILITIES
# ============================================================

def seconds_to_hms(s):
    h = int(s // 3600); m = int((s % 3600) // 60); s = s % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

def count_tokens(text):
    if not isinstance(text, str) or not text: return 0
    return len(_enc.encode(text))

def clean_text(text, max_chars=12000):
    if text is None: return ""
    text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"[\x01-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    return text[:max_chars] if len(text) > max_chars else text

def is_valid_embedding_text(text):
    return isinstance(text, str) and text.strip() != ""

def safe_placeholder_text():
    return "[invalid or empty text]"

def prepare_embedding_text(text):
    cleaned = clean_text(text, max_chars=12000)
    return cleaned if is_valid_embedding_text(cleaned) else safe_placeholder_text()

def safe_json_extract(raw):
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def prompt_hash(prompt):
    return hashlib.md5(prompt.encode()).hexdigest()[:8]

def hint_entropy(hints):
    vals = np.clip([hints[c] for c in LABEL_COLS], 1e-9, 1 - 1e-9)
    return float(-np.sum(vals * np.log(vals) + (1 - vals) * np.log(1 - vals)))

def contains_toxic_keyword(text):
    t = clean_text(text).lower()
    return any(re.search(r"\b" + re.escape(kw) + r"\b", t) for kw in TOXIC_KEYWORDS)

def word_count(text):
    return len(str(text).split())

def contains_group_noun(text):
    """FIX 3: Detect group-identity nouns for identity_hate hint boosting."""
    t = clean_text(text).lower()
    return any(re.search(r"\b" + re.escape(g) + r"\b", t) for g in GROUP_IDENTITY_NOUNS)


# ============================================================
# RETRY WRAPPER
# ============================================================

def call_with_backoff(fn, *args, max_retries=5, base_delay=1.0, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower() or "timeout" in err.lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [backoff] attempt {attempt+1} — {delay:.1f}s ({err[:60]})")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")

async def async_backoff(fn, *args, max_retries=5, base_delay=1.0, **kwargs):
    for attempt in range(max_retries):
        try:
            return await fn(*args, **kwargs)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower() or "timeout" in err.lower():
                await asyncio.sleep(base_delay * (2 ** attempt))
            else:
                raise
    raise RuntimeError("Max retries exceeded")


# ============================================================
# EMBEDDING
# ============================================================

def embed_batch(texts, model=EMBED_MODEL, batch_size=EMBED_BATCH):
    vecs = []
    total_tokens = 0
    total_batches = math.ceil(len(texts) / batch_size)
    t0 = time.perf_counter()

    print("=" * 72)
    print("EMBEDDING")
    print("=" * 72)

    for batch_num, i in enumerate(range(0, len(texts), batch_size), 1):
        raw_batch = texts[i:i + batch_size]
        batch = [prepare_embedding_text(t).replace("\n", " ") for t in raw_batch]
        batch_tokens = sum(count_tokens(t) for t in batch)
        total_tokens += batch_tokens

        try:
            resp = call_with_backoff(client.embeddings.create, model=model, input=batch)
            vecs.extend([x.embedding for x in resp.data])
        except Exception:
            print(f"\nBATCH FAILED: {batch_num}/{total_batches} — falling back one-by-one")
            for local_idx, single_text in enumerate(batch):
                global_idx = i + local_idx
                try:
                    resp = call_with_backoff(client.embeddings.create, model=model, input=[single_text])
                    vecs.append(resp.data[0].embedding)
                except Exception as inner_e:
                    print(f"  BAD ROW {global_idx} — placeholder. Error: {inner_e}")
                    fallback = safe_placeholder_text()
                    resp = call_with_backoff(client.embeddings.create, model=model, input=[fallback])
                    vecs.append(resp.data[0].embedding)

        done = min(i + batch_size, len(texts))
        pct  = 100.0 * done / len(texts)
        cost = (batch_tokens / 1000) * EMBED_COST_PER_1K
        print(f"  [Batch {batch_num:>3}/{total_batches}] {done:>7,}/{len(texts):<7,} ({pct:6.2f}%) | "
              f"tokens~{batch_tokens:>8,} | cost~${cost:.6f}")

    arr = np.array(vecs, dtype=np.float32)
    arr /= np.linalg.norm(arr, axis=1, keepdims=True) + 1e-12

    elapsed = time.perf_counter() - t0
    total_cost = (total_tokens / 1000) * EMBED_COST_PER_1K
    print(f"\n  Embedded {len(texts):,} | {total_tokens:,} tokens | ${total_cost:.6f} | {seconds_to_hms(elapsed)}")
    print("=" * 72)
    return arr, total_tokens, total_cost, elapsed

def embed_one(text):
    cleaned = prepare_embedding_text(text).replace("\n", " ")
    try:
        resp = call_with_backoff(client.embeddings.create, model=EMBED_MODEL, input=[cleaned])
    except Exception:
        cleaned = safe_placeholder_text()
        resp = call_with_backoff(client.embeddings.create, model=EMBED_MODEL, input=[cleaned])
    v = np.array(resp.data[0].embedding, dtype=np.float32)
    v /= np.linalg.norm(v) + 1e-12
    return v, count_tokens(cleaned)


# ============================================================
# PURITY & LOCAL CENTROID
# ============================================================

def label_purity(row_dict, target_label):
    active = sum(row_dict[c] for c in LABEL_COLS if row_dict[c] == 1)
    if active == 0 or row_dict[target_label] != 1: return 0.0
    return 1.0 / active

def select_pure_examples(df, target_label, purity_floor, max_examples=200):
    subset = df[df[target_label] == 1].copy()
    subset["_purity"] = subset.apply(lambda r: label_purity(r.to_dict(), target_label), axis=1)
    return subset[subset["_purity"] >= purity_floor].sort_values("_purity", ascending=False).head(max_examples)

def build_local_centroids(hard_emb, clean_emb, k=LOCAL_CENTROID_K):
    print(f"  Computing local centroids (k={k}) for {len(hard_emb):,} hard-label rows...")
    sims = hard_emb @ clean_emb.T
    local_centroids = np.zeros_like(hard_emb)
    for i in range(len(hard_emb)):
        top_k = np.argsort(-sims[i])[:k]
        lc = clean_emb[top_k].mean(axis=0)
        lc /= np.linalg.norm(lc) + 1e-12
        local_centroids[i] = lc
    return local_centroids

def topic_extremeness(hard_emb, local_centroids):
    return 1.0 - np.einsum("ij,ij->i", hard_emb, local_centroids)


# ============================================================
# CURATION SHORTLIST
# ============================================================

def export_curation_shortlist(df):
    print("\n── Generating curation shortlist ──")
    shortlist = {}
    for label, max_ex in [("severe_toxic", 60), ("threat", 40), ("identity_hate", 40)]:
        pure_df = select_pure_examples(df, label, purity_floor=PURITY_FLOOR[label], max_examples=max_ex)
        shortlist[label] = []
        for _, row in pure_df.iterrows():
            combo = "+".join(c for c in LABEL_COLS if row[c] == 1)
            shortlist[label].append({
                "comment_text": row["comment_text"],
                "label_combo":  combo,
                "n_labels":     int(row["n_labels"]),
                "purity":       round(row["_purity"], 3),
                "keep":         None,
                "note":         ""
            })
        print(f"  [{label}] {len(shortlist[label])} candidates (purity >= {PURITY_FLOOR[label]})")
    with open(CURATED_FILE, "w", encoding="utf-8") as f:
        json.dump(shortlist, f, ensure_ascii=False, indent=2)
    print(f"\n  Saved → {CURATED_FILE}")

def load_curated_examples():
    if not os.path.exists(CURATED_FILE): return {}
    with open(CURATED_FILE, "r", encoding="utf-8") as f:
        raw = json.load(f)
    curated = {}
    for label, entries in raw.items():
        kept = [e for e in entries if e.get("keep") is True]
        if kept:
            curated[label] = kept
            print(f"  Loaded {len(kept)} curated examples for [{label}]")
    return curated


# ============================================================
# BUILD INDEX
# ============================================================

def build_index():
    t0 = time.perf_counter()
    print("=" * 72)
    print("BUILD INDEX v5")
    print("=" * 72)

    df = pd.read_csv(FEW_SHOT_CSV).copy()
    df = df[df["comment_text"].notna()].copy()
    df["comment_text"] = df["comment_text"].astype(str)
    df["embed_text"]   = df["comment_text"].map(prepare_embedding_text)

    for c in LABEL_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    df["n_labels"]      = df[LABEL_COLS].sum(axis=1)
    df["is_clean"]      = (df["n_labels"] == 0).astype(np.int8)
    df["is_multilabel"] = (df["n_labels"] > 1).astype(np.int8)
    df["label_str"]     = df[LABEL_COLS].apply(
        lambda r: ",".join(c for c in LABEL_COLS if r[c] == 1) or "clean", axis=1
    )
    df["text_hash"] = df["comment_text"].apply(lambda t: hashlib.md5(str(t).encode()).hexdigest()[:16])

    print(f"Rows: {len(df):,}")
    print("Label distribution:")
    print(df[LABEL_COLS + ["is_clean"]].sum().to_string())

    # Purify clean centroid
    clean_mask = df["is_clean"] == 1
    contaminated = df[clean_mask]["embed_text"].apply(contains_toxic_keyword)
    contaminated = contaminated.reindex(df.index, fill_value=False)
    pure_clean_mask = clean_mask & ~contaminated
    print(f"\n  Pure clean: {pure_clean_mask.sum():,} (contaminated excluded: {contaminated.sum():,})")

    all_emb, total_tokens, total_cost, embed_time = embed_batch(df["embed_text"].tolist())

    clean_emb = all_emb[pure_clean_mask.values]
    hard_label_mask = df[HARD_LABELS].any(axis=1)
    hard_idx = np.where(hard_label_mask.values)[0]
    hard_emb = all_emb[hard_idx]

    local_cents = build_local_centroids(hard_emb, clean_emb)
    topic_ext   = topic_extremeness(hard_emb, local_cents)

    df["topic_extremeness"] = np.nan
    df.iloc[hard_idx, df.columns.get_loc("topic_extremeness")] = topic_ext
    df["topic_extremeness"] = df["topic_extremeness"].fillna(0.0)

    for label in HARD_LABELS:
        sub = df[df[label] == 1].copy()
        sub["_p"] = sub.apply(lambda r: label_purity(r.to_dict(), label), axis=1)
        df.loc[df[label] == 1, f"purity_{label}"] = sub["_p"]
    for label in HARD_LABELS:
        if f"purity_{label}" not in df.columns:
            df[f"purity_{label}"] = 0.0
        df[f"purity_{label}"] = df[f"purity_{label}"].fillna(0.0)

    export_curation_shortlist(df)
    curated = load_curated_examples()
    curated_flags = np.zeros(len(df), dtype=np.int8)
    for label, entries in curated.items():
        for entry in entries:
            match = df[df["comment_text"] == entry["comment_text"]]
            if len(match) > 0:
                curated_flags[df.index.get_loc(match.index[0])] = 1
    df["is_curated"] = curated_flags

    # Save
    save_dict = {
        "embeddings":        all_emb,
        "is_clean":          df["is_clean"].values.astype(np.int8),
        "is_multilabel":     df["is_multilabel"].values.astype(np.int8),
        "is_curated":        df["is_curated"].values.astype(np.int8),
        "topic_extremeness": df["topic_extremeness"].values.astype(np.float32),
    }
    for c in LABEL_COLS:
        save_dict[c] = df[c].values.astype(np.int8)
    for label in HARD_LABELS:
        save_dict[f"purity_{label}"] = df[f"purity_{label}"].values.astype(np.float32)

    np.savez_compressed(INDEX_FILE, **save_dict)

    meta_cols = (
        ["comment_text","embed_text","label_str","n_labels",
         "is_clean","is_multilabel","is_curated","topic_extremeness","text_hash"]
        + LABEL_COLS
        + [f"purity_{l}" for l in HARD_LABELS]
    )
    with open(META_FILE, "w", encoding="utf-8") as f:
        json.dump(df[meta_cols].to_dict(orient="records"), f, ensure_ascii=False)

    elapsed = time.perf_counter() - t0
    print(f"\nSaved: {INDEX_FILE}  +  {META_FILE}")
    print(f"Total tokens: {total_tokens:,}  Cost: ${total_cost:.6f}  Time: {seconds_to_hms(elapsed)}")
    print("=" * 72)


# ============================================================
# LOAD INDEX
# ============================================================

def load_index():
    z = np.load(INDEX_FILE)
    pool_emb       = z["embeddings"]
    is_clean_arr   = z["is_clean"]
    is_curated_arr = z["is_curated"]
    topic_ext_arr  = z["topic_extremeness"]
    label_arrays   = {c: z[c] for c in LABEL_COLS}
    purity_arrays  = {l: z[f"purity_{l}"] for l in HARD_LABELS}
    with open(META_FILE, "r", encoding="utf-8") as f:
        pool_meta = json.load(f)
    return pool_emb, is_clean_arr, is_curated_arr, topic_ext_arr, label_arrays, purity_arrays, pool_meta


# ============================================================
# RETRIEVAL — HINTS, REGEX, SKIP-GATE
# ============================================================

def infer_neighbor_hints(sim, label_arrays, topn=40):
    idx = np.argsort(-sim)[:topn]
    w   = np.maximum(sim[idx], 0)
    out = {}
    for c in LABEL_COLS:
        out[c] = float((w * label_arrays[c][idx]).sum() / w.sum()) if w.sum() > 0 else 0.0

    # FIX 3: boost identity_hate hint when group nouns detected in neighbours
    # (handled at call site using contains_group_noun)
    return out


def regex_cues(text):
    """
    Improved regex cues with:
    - FIX 2: threat requires second-person target proximity
    - FIX 3: expanded coded identity-hate patterns
    - Deobfuscation layer
    - Sarcasm detection
    """
    t = clean_text(text).lower()

    # Deobfuscation
    t_deob = (t
        .replace("@", "a").replace("0", "o").replace("1", "i").replace("3", "e")
        .replace("4", "a").replace("5", "s").replace("7", "t").replace("|", "i")
        .replace("!", "i").replace("$", "s")
    )

    def no_negation(pattern, window=60):
        for m in re.finditer(pattern, t):
            prec = t[max(0, m.start() - window):m.start()]
            if re.search(r"\b(not|never|don'?t|didn'?t|won'?t|no one|nobody|cannot|can'?t|wouldn'?t)\b", prec):
                return False
            return True
        return False

    sarcasm_signal = bool(re.search(
        r"\b(oh sure|yeah right|totally not|great job|nice work|brilliant|so helpful|wow thanks)"
        r"|\.\.\.|/s\b", t
    ))

    cues = {
        "obscene": int(bool(re.search(
            r"\b(fuck(?:ing)?|motherfucker|shit(?:ty)?|bitch|asshole|cunt|dick(?:head)?|pussy)\b",
            t_deob))),

        # FIX 2: threat now requires (a) action verb + harm verb with target proximity,
        # OR (b) direct "kill yourself" style patterns
        "threat": int(
            no_negation(
                r"\b(i(?:'ll| will| am going to| gonna)|you(?:'re| are) going to)\b"
                r".{0,40}\b(kill|shoot|stab|hurt|murder|destroy|burn|hang|beat)\b"
                r".{0,30}\b(you|your|him|her|them)\b"   # FIX 2: target proximity
            )
            or bool(re.search(
                r"\bkill yourself\b|\bgo die\b|\bi hope you die\b"
                r"|\bi('ll| will) find you\b.{0,30}\b(kill|hurt|destroy)\b",
                t))
        ),

        # FIX 3: expanded identity_hate patterns with dog-whistles
        "identity_hate": int(bool(re.search(
            r"\b(all\s+(?:muslims?|jews?|blacks?|whites?|gays?|immigrants?|refugees?|mexicans?|asians?|arabs?)\s+(?:are|should|must|deserve|need to))\b"
            r"|\b(f+a+g+s?|n+i+g+g+[ae]r|k[iy]k[ey]|sp[i]+c|ch[i]+nk|w+e+t+b+a+c+k|go+k)\b"
            r"|\b(these people|those people|you people)\s.{0,40}(country|here|belong|go back|leave)\b"
            r"|\b(go back to|deport all|ban all)\s.{0,30}(country|where you came)\b"
            r"|\b(subhuman|cockroach|vermin|animal|ape|monkey)\b.{0,40}"
            r"\b(muslim|jew|black|immigrant|mexican|arab|african)\b",
            t_deob))),

        "insult": int(bool(re.search(
            r"\b(you(?:'re| are) (?:an? )?(?:idiot|moron|stupid|loser|pig|trash|scum|pathetic|worthless|dumb|brain.?dead))\b"
            r"|\b(what an? (?:idiot|moron|loser|joke))\b"
            r"|\b(shut (?:the fuck )?up)\b",
            t))),
    }
    cues["toxic"]        = int(any(cues.values()))
    cues["severe_toxic"] = int(cues["threat"] or bool(re.search(
        r"\b(subhuman|vermin|filthy animal|piece of garbage|should not exist|exterminate)\b", t)))
    cues["sarcasm"]      = int(sarcasm_signal)
    return cues


def should_skip_gpt(text, sim, is_clean_arr, hints, cues):
    """
    FIX 4: More aggressive skip for genuinely clean comments.
    Target: 8-12% skip rate (up from 0.97%).
    """
    # Never skip short comments
    if word_count(text) <= SHORT_COMMENT_WORDS:
        return False

    # Hard label override
    for label in HARD_LABELS:
        if hints[label] >= HARD_LABEL_HINT_FLOOR or cues.get(label, 0) == 1:
            return False

    # Soft guard
    for label in SOFT_GUARD_LABELS:
        if hints[label] >= SOFT_GUARD_FLOOR:
            return False

    # Never skip if high obscenity + insult combo
    if hints.get("obscene", 0) >= 0.12 and hints.get("insult", 0) >= 0.08:
        return False

    # Never skip if sarcasm detected
    if cues.get("sarcasm", 0) == 1:
        return False

    # Never skip if contains group identity nouns (FIX 3)
    if contains_group_noun(text):
        return False

    # Never skip if contains toxic keywords
    if contains_toxic_keyword(text):
        return False

    top_idx     = np.argsort(-sim)[:8]
    clean_ratio = float(is_clean_arr[top_idx].mean()) if len(top_idx) > 0 else 0.0
    top1        = float(sim[top_idx[0]]) if len(top_idx) > 0 else 0.0

    no_hint  = all(hints[c] < SKIP_ALL_HINT_FLOOR for c in LABEL_COLS)
    no_regex = sum(v for k, v in cues.items() if k != "sarcasm") == 0

    # FIX 4: Ultra-clean fast path — all 8 neighbours clean + no signals
    ultra_clean = (clean_ratio == 1.0) and no_hint and no_regex

    if ultra_clean and top1 >= 0.85 and word_count(text) > 20:
        return True

    # Standard skip path (relaxed thresholds)
    return (top1 >= SKIP_TOP1_SIM) and (clean_ratio >= SKIP_CLEAN_RATIO) and no_hint and no_regex


def get_top(mask, score, used_hashes, pool_meta, k=1):
    idx = np.where(mask)[0]
    if len(idx) == 0: return []
    order = idx[np.argsort(-score[idx])]
    out = []
    for i in order:
        i = int(i)
        h = pool_meta[i].get("text_hash", str(i))
        if h not in used_hashes:
            used_hashes.add(h)
            out.append(i)
            if len(out) >= k: break
    return out


def hard_label_score(sim, purity_arrays, topic_ext_arr, label,
                     alpha=0.5, beta=0.3, gamma=0.2):
    purity = purity_arrays[label].astype(np.float32)
    te = topic_ext_arr.astype(np.float32)
    te_pos = te[te > 0]
    if len(te_pos) > 0:
        te_norm = np.clip((te - te_pos.min()) / (te_pos.max() - te_pos.min() + 1e-12), 0, 1)
    else:
        te_norm = np.zeros_like(te)
    return alpha * sim + beta * purity + gamma * te_norm


def retrieve_combo_diverse(sim, label_arrays, pool_meta, used_hashes, hints, k=2):
    chosen = []
    seen_combos = set()
    active_labels = [c for c in LABEL_COLS if hints[c] >= 0.08]
    if not active_labels: return chosen

    top_idx = np.argsort(-sim)[:200]
    for i in top_idx:
        i = int(i)
        h = pool_meta[i].get("text_hash", str(i))
        if h in used_hashes: continue
        combo = pool_meta[i].get("label_str", "clean")
        if combo not in seen_combos:
            seen_combos.add(combo)
            used_hashes.add(h)
            chosen.append(i)
            if len(chosen) >= k: break
    return chosen


def retrieve_examples(text, pool_emb, is_clean_arr, is_curated_arr,
                      topic_ext_arr, label_arrays, purity_arrays, pool_meta):
    qvec, embed_tokens = embed_one(text)
    sim     = pool_emb @ qvec
    hints   = infer_neighbor_hints(sim, label_arrays, topn=40)
    cues    = regex_cues(text)
    entropy = hint_entropy(hints)

    # FIX 3: Boost identity_hate hint when group nouns present
    if contains_group_noun(text) and hints["identity_hate"] < 0.15:
        hints["identity_hate"] = max(hints["identity_hate"], 0.12)

    k_total = RETRIEVE_K_HIGH if entropy > 3.0 else RETRIEVE_K
    used_hashes = set()
    chosen  = []

    # 1) Curated examples first
    for label in HARD_LABELS:
        if hints[label] >= 0.10 or cues.get(label, 0) == 1:
            curated_mask = (is_curated_arr == 1) & (label_arrays[label] == 1)
            if curated_mask.any():
                score = hard_label_score(sim, purity_arrays, topic_ext_arr, label)
                chosen += get_top(curated_mask, score, used_hashes, pool_meta, k=1)

    # 2) Best overall + best toxic
    chosen += get_top(np.ones(len(sim), dtype=bool), sim, used_hashes, pool_meta, k=1)
    chosen += get_top(label_arrays["toxic"] == 1, sim, used_hashes, pool_meta, k=1)

    # 3) Purity-ranked hard positive + contrast example
    for label in HARD_LABELS:
        if hints[label] >= 0.10 or cues.get(label, 0) == 1:
            score = hard_label_score(sim, purity_arrays, topic_ext_arr, label)
            chosen += get_top(label_arrays[label] == 1, score, used_hashes, pool_meta, k=1)
            # Contrast: same topic but label not set
            chosen += get_top(
                (label_arrays["toxic"] == 1) & (label_arrays[label] == 0),
                sim, used_hashes, pool_meta, k=1
            )

    # 4) Gradient pairs for ALL hard labels
    for label in HARD_LABELS:
        if hints[label] >= 0.08 or cues.get(label, 0) == 1:
            score = hard_label_score(sim, purity_arrays, topic_ext_arr, label)
            chosen += get_top(label_arrays[label] == 1, score, used_hashes, pool_meta, k=1)
            chosen += get_top(
                (label_arrays["toxic"] == 1) & (label_arrays[label] == 0),
                sim, used_hashes, pool_meta, k=1
            )

    # 5) FIX 3: If identity_hate is hinted, ensure at least one identity_hate example
    if (hints["identity_hate"] >= 0.08 or cues.get("identity_hate", 0) == 1):
        has_ih = any(pool_meta[i].get("identity_hate", 0) == 1 for i in chosen if i < len(pool_meta))
        if not has_ih:
            chosen += get_top(label_arrays["identity_hate"] == 1, sim, used_hashes, pool_meta, k=1)

    # 6) Combo-diverse fill
    if len(chosen) < k_total:
        chosen += retrieve_combo_diverse(sim, label_arrays, pool_meta, used_hashes, hints,
                                         k=k_total - len(chosen))

    # 7) Similarity fill
    if len(chosen) < k_total:
        chosen += get_top(np.ones(len(sim), dtype=bool), sim, used_hashes, pool_meta,
                          k=k_total - len(chosen))

    chosen = chosen[:k_total]

    examples = []
    for i in chosen:
        row = pool_meta[i]
        labels = [c for c in LABEL_COLS if row[c] == 1]
        examples.append({
            "comment_text": row["comment_text"],
            "label_str":    ", ".join(labels) if labels else "clean",
            "similarity":   float(sim[i]),
            "is_curated":   int(row.get("is_curated", 0)),
        })

    return sim, hints, cues, examples, embed_tokens, entropy


# ============================================================
# GPT PREDICTION — FIX 1, 2, 3, 7: IMPROVED PROMPT
# ============================================================

def format_examples(examples):
    blocks = []
    for i, ex in enumerate(examples, 1):
        curated_tag = " [curated]" if ex.get("is_curated") else ""
        blocks.append(
            f"Example {i}{curated_tag}\n"
            f"Comment: {ex['comment_text']}\n"
            f"Labels:  {ex['label_str']}"
        )
    return "\n\n".join(blocks)


def build_prompt(comment_text, examples, rephrased=False):
    """
    FIX 1: severe_toxic gets a 4-point severity rubric.
    FIX 2: threat explicitly excludes hyperbole.
    FIX 3: identity_hate boundary clarified.
    FIX 7: Prompt addresses common LLM failure modes.
    """
    intro = (
        "Re-evaluate this comment carefully, paying extra attention to severity and context:"
        if rephrased else
        "You are an expert multi-label toxicity classifier."
    )

    if COT_ENABLED:
        output_instruction = (
            "First write 1-2 sentences of reasoning, then output EXACTLY this JSON "
            "with confidence scores 0-100:"
        )
        json_template = (
            '{\n  "reasoning": "...",\n  "scores": {\n'
            '    "toxic": 0,\n    "severe_toxic": 0,\n    "obscene": 0,\n'
            '    "threat": 0,\n    "insult": 0,\n    "identity_hate": 0\n  }\n}'
        )
    else:
        output_instruction = (
            "Output ONLY this JSON with confidence scores 0-100 and nothing else:"
        )
        json_template = (
            '{\n  "scores": {\n'
            '    "toxic": 0,\n    "severe_toxic": 0,\n    "obscene": 0,\n'
            '    "threat": 0,\n    "insult": 0,\n    "identity_hate": 0\n  }\n}'
        )

    return f"""{intro}

Labels and definitions (read carefully — common mistakes noted):

- toxic: rude, hateful, or aggressive language that could make someone want to leave a conversation.

- severe_toxic: SIGNIFICANTLY more intense than ordinary toxic. Use this severity scale:
  * Mild toxic (score 10-30): single profanity, casual rudeness, mild name-calling
  * Moderate toxic (score 30-50): repeated profanity, aggressive hostility, personal attacks
  * Severe toxic (score 50-80): sustained extreme abuse, dehumanising language, violent hostility,
    intense degrading attacks, or repeated extreme obscene abuse directed at a person
  * Extreme toxic (score 80-100): explicit calls for harm combined with dehumanisation
  COMMON MISTAKE: Do not require genocide-level language for severe_toxic. Intense, sustained
  personal abuse with extreme profanity IS severe_toxic even without extermination language.
  ALSO: A single profane word alone is NOT severe_toxic — severity requires intensity OR persistence.

- obscene: explicit profanity, vulgar sexual language, or graphic crude language.

- threat: a CREDIBLE, SPECIFIC indication of intent to cause physical harm to a person.
  REQUIRES: (a) an identifiable target AND (b) a stated harmful action.
  IS a threat: "I will find you and kill you", "you're going to get shot"
  IS NOT a threat: "I could kill for a pizza" (hyperbole), "fuck off and die" (anger expression),
  "someone should teach them a lesson" (vague, no specific target or action),
  "I hope you get hit by a bus" (wish, not stated intent to act).
  COMMON MISTAKE: Angry language is NOT a threat unless it contains specific intent to harm.

- insult: a direct personal attack on someone's character, intelligence, or worth.
  Must target a specific person or addressee, not a general statement.

- identity_hate: attacks targeting a person BECAUSE OF their membership in a protected group
  (race, religion, ethnicity, gender, sexual orientation, nationality, disability).
  IS identity_hate: "all Muslims are terrorists", "go back to your country", slurs targeting a group
  IS NOT identity_hate: "you're an idiot" (personal insult, not group-based),
  "I disagree with Islam" (critique of religion, not attack on people)
  COMMON MISTAKE: Missing coded/indirect hate speech like "these people don't belong here"
  when referring to an ethnic or religious group.

Decision rules:
1. Multi-label output is expected — most toxic comments have 2-4 labels active.
2. If severe_toxic is positive, toxic must also be positive.
3. Score each label independently based on its definition above.
4. Pay attention to the INTENSITY and PERSISTENCE of language, not just the presence of bad words.

Reference examples (study the severity gradients):
{format_examples(examples)}

Classify this comment:
{comment_text}

{output_instruction}
{json_template}""".strip()


def build_severity_check_prompt(comment_text):
    """
    FIX 1: Dedicated second-pass prompt for severe_toxic borderline cases.
    Only called when initial toxic score is high but severe_toxic is borderline.
    """
    return f"""You are evaluating whether a toxic comment reaches the level of SEVERE toxicity.

Severity scale:
- NOT severe: single profanity, mild rudeness, casual insult, disagreement expressed rudely
- BORDERLINE: aggressive hostility, multiple strong profanities, demeaning personal attack
- SEVERE: sustained extreme abuse, dehumanising language, intense degrading attack, violent hostility, or repeated extreme obscenity directed at a person

Comment: {comment_text}

Rate the severity as a single integer 0-100 where:
0-30 = not severe, 30-60 = borderline, 60-100 = clearly severe.

Output ONLY this JSON:
{{"severe_toxic_score": 0}}""".strip()


def build_identity_check_prompt(comment_text):
    """
    FIX 3: Dedicated second-pass prompt for identity_hate borderline cases.
    """
    return f"""Does this comment attack, demean, or express hatred toward people BECAUSE OF
their membership in a protected group (race, religion, ethnicity, gender, sexual orientation,
nationality, disability)?

Look for:
- Explicit slurs targeting a group
- Stereotyping or generalising negatively about a group
- Coded language like "these people", "go back to your country"
- Dehumanising comparisons applied to a group

Comment: {comment_text}

Rate the likelihood of identity-based hate as a single integer 0-100.

Output ONLY this JSON:
{{"identity_hate_score": 0}}""".strip()


def gpt_call_raw(prompt, seed=GPT_SEED):
    resp = call_with_backoff(
        client.chat.completions.create,
        model=GPT_MODEL,
        temperature=0,
        seed=seed,
        messages=[
            {"role": "system", "content": "You are a precise toxicity classification assistant. Output only valid JSON."},
            {"role": "user",   "content": prompt}
        ]
    )
    raw     = resp.choices[0].message.content.strip()
    in_tok  = count_tokens(prompt)
    out_tok = count_tokens(raw)
    return raw, in_tok, out_tok


def extract_scores(raw):
    parsed = safe_json_extract(raw)
    if parsed is None: return None
    scores = parsed.get("scores", parsed)
    if not isinstance(scores, dict): return None
    result = {}
    for c in LABEL_COLS:
        try: result[c] = int(float(scores.get(c, 0)))
        except (ValueError, TypeError): result[c] = 0
    if all(v in (0, 1) for v in result.values()):
        result = {c: v * 100 for c, v in result.items()}
    return result


def apply_thresholds(scores, thresholds=None):
    t = thresholds or LABEL_THRESHOLDS
    return {c: int(scores[c] >= t[c]) for c in LABEL_COLS}


def is_borderline_hard_label(scores):
    """FIX 6: Only check borderlines for hard labels (the problematic ones)."""
    return any(
        BORDERLINE_LOW < scores[c] < BORDERLINE_HIGH and scores[c] > 15
        for c in CONSISTENCY_LABELS
    )


def predict_gpt(comment_text, examples, thresholds=None, hints=None, cues=None):
    prompt = build_prompt(comment_text, examples)
    raw, in_tok, out_tok = gpt_call_raw(prompt)
    scores = extract_scores(raw)

    if scores is None:
        fallback = (
            f"Classify this comment for toxicity. Return ONLY valid JSON with integer scores 0-100:\n"
            f"{comment_text}\n\n"
            f'{{"scores":{{"toxic":0,"severe_toxic":0,"obscene":0,"threat":0,"insult":0,"identity_hate":0}}}}'
        )
        raw2, in2, out2 = gpt_call_raw(fallback, seed=GPT_SEED + 1)
        in_tok += in2; out_tok += out2
        scores = extract_scores(raw2) or {c: 0 for c in LABEL_COLS}
        raw = raw2

    used_consistency = False
    extra_passes = []

    # FIX 1: Severity check for borderline severe_toxic
    # If toxic is high but severe_toxic is in the ambiguous zone
    if scores["toxic"] >= 40 and 20 <= scores["severe_toxic"] <= 65:
        sev_prompt = build_severity_check_prompt(comment_text)
        sev_raw, sev_in, sev_out = gpt_call_raw(sev_prompt, seed=GPT_SEED + 50)
        in_tok += sev_in; out_tok += sev_out
        sev_parsed = safe_json_extract(sev_raw)
        if sev_parsed and "severe_toxic_score" in sev_parsed:
            sev_score = int(float(sev_parsed["severe_toxic_score"]))
            # Weighted merge: 0.5 original + 0.5 severity check
            scores["severe_toxic"] = int(0.5 * scores["severe_toxic"] + 0.5 * sev_score)
        extra_passes.append(f"SEV_CHECK: {sev_raw}")

    # FIX 3: Identity check for borderline identity_hate when group nouns present
    if (hints and hints.get("identity_hate", 0) >= 0.08 and
            15 <= scores["identity_hate"] <= 55):
        ih_prompt = build_identity_check_prompt(comment_text)
        ih_raw, ih_in, ih_out = gpt_call_raw(ih_prompt, seed=GPT_SEED + 60)
        in_tok += ih_in; out_tok += ih_out
        ih_parsed = safe_json_extract(ih_raw)
        if ih_parsed and "identity_hate_score" in ih_parsed:
            ih_score = int(float(ih_parsed["identity_hate_score"]))
            scores["identity_hate"] = int(0.5 * scores["identity_hate"] + 0.5 * ih_score)
        extra_passes.append(f"IH_CHECK: {ih_raw}")

    # FIX 6: Adaptive self-consistency — only for hard labels in borderline zone
    if is_borderline_hard_label(scores):
        prompt2 = build_prompt(comment_text, examples, rephrased=True)
        raw2, in2, out2 = gpt_call_raw(prompt2, seed=GPT_SEED + 99)
        scores2 = extract_scores(raw2) or {c: 0 for c in LABEL_COLS}

        # FIX 6: Weighted merge for hard labels; simple average for others
        for c in LABEL_COLS:
            if c in CONSISTENCY_LABELS:
                scores[c] = CONSISTENCY_WEIGHT_PASS1 * scores[c] + CONSISTENCY_WEIGHT_PASS2 * scores2[c]
            else:
                scores[c] = (scores[c] + scores2[c]) / 2
        scores = {c: int(v) for c, v in scores.items()}

        in_tok += in2; out_tok += out2
        extra_passes.append(f"CONSISTENCY: {raw2}")
        used_consistency = True

    if extra_passes:
        raw = f"MAIN: {raw}\n\n" + "\n\n".join(extra_passes)

    pred = apply_thresholds(scores, thresholds)
    return pred, scores, raw, in_tok, out_tok, prompt_hash(prompt), used_consistency


def predict_knn_clean():
    return (
        {c: 0 for c in LABEL_COLS},
        {c: 0 for c in LABEL_COLS},
        "skipped_gpt_high_confidence_clean",
        0, 0, "skipped", False
    )


# ============================================================
# POST-PROCESSING — FIX 5: LOGICALLY CONSISTENT RULES
# ============================================================

def calibrate_predictions(pred, scores):
    """
    FIX 5: Logically consistent heuristic rules.

    Rules applied:
      R1. Label hierarchy: if any sub-label is 1, toxic must be 1.
      R2. Rare-label suppression: if toxic score < 15 (raised from 25),
          zero severe_toxic/threat/identity_hate.
      R3. Insult smoothing: if insult=1 but toxic score < 20, flip insult to 0.
      R4. Severity floor (NEW): if obscene>=80 AND insult>=70 AND severe_toxic
          was scored <40, boost severe_toxic score to 40 (intense abuse indicator).
      R5. Identity→insult hierarchy (NEW): if identity_hate=1, ensure insult=1.
    """
    if not APPLY_HEURISTIC_RULES:
        return pred

    # R4: Severity floor — intense obscene abuse should be flagged as potentially severe
    if scores.get("obscene", 0) >= 80 and scores.get("insult", 0) >= 70:
        if scores.get("severe_toxic", 0) < 40:
            scores["severe_toxic"] = 40

    # R1: hierarchy
    for sub in ["severe_toxic", "threat", "identity_hate", "insult", "obscene"]:
        if pred[sub] == 1:
            pred["toxic"] = 1

    # R2: rare label suppression (FIX 5: threshold raised to avoid contradiction with toxic threshold=20)
    if scores["toxic"] < 15:
        pred["severe_toxic"] = 0
        pred["threat"]       = 0
        pred["identity_hate"] = 0

    # R3: insult smoothing
    if pred["insult"] == 1 and scores["toxic"] < 20:
        pred["insult"] = 0

    # R5: identity_hate → insult hierarchy
    if pred["identity_hate"] == 1:
        pred["insult"] = 1

    return pred


# ============================================================
# ASYNC INFERENCE
# ============================================================

async def process_row_async(row, pool_emb, is_clean_arr, is_curated_arr,
                            topic_ext_arr, label_arrays, purity_arrays,
                            pool_meta, semaphore, thresholds=None):
    loop = asyncio.get_event_loop()
    row_start = time.perf_counter()

    sim, hints, cues, examples, embed_tokens, entropy = await loop.run_in_executor(
        None, retrieve_examples,
        row.comment_text, pool_emb, is_clean_arr, is_curated_arr,
        topic_ext_arr, label_arrays, purity_arrays, pool_meta
    )

    if should_skip_gpt(row.comment_text, sim, is_clean_arr, hints, cues):
        pred, scores, raw, in_tok, out_tok, phash, used_cons = predict_knn_clean()
        skipped = True
    else:
        async with semaphore:
            pred, scores, raw, in_tok, out_tok, phash, used_cons = await loop.run_in_executor(
                None, predict_gpt, row.comment_text, examples, thresholds, hints, cues
            )
        skipped = False

    pred = calibrate_predictions(pred, scores)

    row_elapsed = time.perf_counter() - row_start

    out = {
        "id":                 str(row.id),
        "comment_text":       row.comment_text,
        "raw_response":       raw,
        "retrieved_examples": json.dumps(examples, ensure_ascii=False),
        "hints":              json.dumps(hints, ensure_ascii=False),
        "cues":               json.dumps(cues, ensure_ascii=False),
        "hint_entropy":       entropy,
        "used_consistency":   used_cons,
        "prompt_version":     phash,
        "skipped_gpt":        skipped,
        "scores":             json.dumps(scores, ensure_ascii=False),
        "cot_enabled":        COT_ENABLED,
        "heuristic_rules":    APPLY_HEURISTIC_RULES,
        "embed_tokens":       embed_tokens,
        "gpt_input_tokens":   in_tok,
        "gpt_output_tokens":  out_tok,
        "row_runtime_sec":    row_elapsed,
    }
    for c in LABEL_COLS:
        out[f"true_{c}"] = int(getattr(row, c))
        out[f"pred_{c}"] = int(pred[c])

    return out


async def run_inference_async(split="dev", thresholds=None):
    t0 = time.perf_counter()
    csv_in  = DEV_CSV        if split == "dev" else HELD_TEST_CSV
    csv_out = OUTPUT_DEV_CSV if split == "dev" else OUTPUT_TEST_CSV

    print("=" * 72)
    print(f"INFERENCE v5  (split={split})  async")
    print(f"  CoT mode          : {'ON' if COT_ENABLED else 'OFF (JSON-only)'}")
    print(f"  Heuristic rules   : {'ON (hybrid)' if APPLY_HEURISTIC_RULES else 'OFF (pure model)'}")
    print(f"  Thresholds source : {'dev-tuned' if thresholds else 'defaults'}")
    print("=" * 72)

    (pool_emb, is_clean_arr, is_curated_arr,
     topic_ext_arr, label_arrays, purity_arrays, pool_meta) = load_index()

    test_df = pd.read_csv(csv_in).dropna(subset=["comment_text"]).copy()
    test_df["comment_text"] = test_df["comment_text"].astype(str)
    for c in LABEL_COLS:
        test_df[c] = pd.to_numeric(test_df[c], errors="coerce").fillna(0).astype(int)
    if "id" not in test_df.columns:
        test_df["id"] = np.arange(len(test_df)).astype(str)
    else:
        test_df["id"] = test_df["id"].astype(str)

    if os.path.exists(csv_out):
        done_df  = pd.read_csv(csv_out)
        done_ids = set(done_df["id"].astype(str))
        rows     = done_df.to_dict(orient="records")
        print(f"Resuming from {len(rows):,} completed rows")
    else:
        done_ids = set()
        rows     = []
        print("Starting new inference run")

    pending = [r for r in test_df.itertuples(index=False) if str(r.id) not in done_ids]
    print(f"Rows to process: {len(pending):,}")

    semaphore = asyncio.Semaphore(ASYNC_CONCURRENCY)
    tasks = [
        process_row_async(
            row, pool_emb, is_clean_arr, is_curated_arr,
            topic_ext_arr, label_arrays, purity_arrays, pool_meta, semaphore, thresholds
        )
        for row in pending
    ]

    completed = 0
    for coro in asyncio.as_completed(tasks):
        result = await coro
        rows.append(result)
        completed += 1
        if completed % 50 == 0:
            pd.DataFrame(rows).to_csv(csv_out, index=False)
            skipped  = sum(1 for r in rows if r.get("skipped_gpt"))
            gpt_calls = len(rows) - skipped
            cost = (
                sum(r.get("embed_tokens", 0) for r in rows) / 1000 * EMBED_COST_PER_1K +
                sum(r.get("gpt_input_tokens", 0) for r in rows) / 1000 * GPT_INPUT_COST_PER_1K +
                sum(r.get("gpt_output_tokens", 0) for r in rows) / 1000 * GPT_OUTPUT_COST_PER_1K
            )
            print(f"[Progress] {len(rows):>6,}/{len(test_df):<6,} | "
                  f"GPT={gpt_calls:>5,} skipped={skipped:>5,} | cost~${cost:.4f}")

    pd.DataFrame(rows).to_csv(csv_out, index=False)

    elapsed   = time.perf_counter() - t0
    skipped   = sum(1 for r in rows if r.get("skipped_gpt"))
    gpt_calls = len(rows) - skipped

    print("=" * 72)
    print(f"INFERENCE SUMMARY ({split})")
    print(f"Rows: {len(rows):,}  GPT: {gpt_calls:,}  Skipped: {skipped:,}  Time: {seconds_to_hms(elapsed)}")
    print(f"Saved: {csv_out}")
    print("=" * 72)


def run_inference(split="dev", thresholds=None):
    try:
        loop = asyncio.get_running_loop()
        if loop.is_running():
            raise RuntimeError("Jupyter detected: use `await run_inference_async(split=...)` instead.")
    except RuntimeError as e:
        if "Jupyter detected" in str(e): raise
    return asyncio.run(run_inference_async(split=split, thresholds=thresholds))


# ============================================================
# FIX 8 — THRESHOLD TUNING WITH RECALL FLOORS
# ============================================================

def tune_thresholds_on_dev(save: bool = True) -> dict:
    """
    FIX 8: Per-label optimisation with recall floors.
    - severe_toxic, threat: optimise F1 with recall >= 0.50
    - identity_hate: optimise F1 with recall >= 0.60
    - Others: standard F1 optimisation
    - Finer sweep: step=2 instead of step=5
    """
    print("=" * 72)
    print("THRESHOLD TUNING ON DEV SET (v5)")
    print("=" * 72)

    if not os.path.exists(OUTPUT_DEV_CSV):
        raise FileNotFoundError(f"{OUTPUT_DEV_CSV} not found. Run run_inference(split='dev') first.")

    df = pd.read_csv(OUTPUT_DEV_CSV)
    y_true = df[[f"true_{c}" for c in LABEL_COLS]].values.astype(int)

    # FIX 8: Per-label recall floors
    RECALL_FLOORS = {
        "severe_toxic":  0.50,
        "threat":        0.50,
        "identity_hate": 0.60,
    }

    best_thresholds = {}
    print(f"\n  {'Label':<20} {'BestT':>6} {'BestF1':>8} {'Prec':>8} {'Rec':>8} {'RecFloor':>10}")
    print("  " + "─" * 66)

    for i, label in enumerate(LABEL_COLS):
        raw_scores = df["scores"].apply(
            lambda s: json.loads(s).get(label, 0) if isinstance(s, str) else 0
        ).values.astype(float)

        recall_floor = RECALL_FLOORS.get(label, 0.0)
        best_t, best_f1, best_p, best_r = LABEL_THRESHOLDS[label], 0.0, 0.0, 0.0

        # FIX 8: Finer granularity sweep
        for t in range(2, 96, 2):
            preds_t = (raw_scores >= t).astype(int)
            p, r, f1, _ = precision_recall_fscore_support(
                y_true[:, i], preds_t, average="binary", zero_division=0
            )
            # Only consider thresholds that meet the recall floor
            if r >= recall_floor and f1 > best_f1:
                best_f1, best_t, best_p, best_r = f1, t, p, r

        # Fallback: if no threshold meets recall floor, use standard F1
        if best_f1 == 0.0:
            for t in range(2, 96, 2):
                preds_t = (raw_scores >= t).astype(int)
                p, r, f1, _ = precision_recall_fscore_support(
                    y_true[:, i], preds_t, average="binary", zero_division=0
                )
                if f1 > best_f1:
                    best_f1, best_t, best_p, best_r = f1, t, p, r

        best_thresholds[label] = best_t
        flag = " *" if best_t != LABEL_THRESHOLDS[label] else ""
        floor_str = f"{recall_floor:.2f}" if recall_floor > 0 else "—"
        print(f"  {label:<20} {best_t:>6} {best_f1:>8.4f} {best_p:>8.4f} {best_r:>8.4f} {floor_str:>10}{flag}")

    print("\n  * = threshold differs from default")

    if save:
        with open(DEV_THRESHOLDS, "w") as f:
            json.dump(best_thresholds, f, indent=2)
        print(f"\n  Saved → {DEV_THRESHOLDS}")

    print("=" * 72)
    return best_thresholds


def load_dev_thresholds() -> dict:
    if not os.path.exists(DEV_THRESHOLDS):
        raise FileNotFoundError(f"{DEV_THRESHOLDS} not found. Run tune_thresholds_on_dev() first.")
    with open(DEV_THRESHOLDS) as f:
        return json.load(f)


# ============================================================
# EVALUATION REPORT
# ============================================================

def evaluate_predictions(split: str = "test"):
    eval_start = time.perf_counter()
    csv_file = OUTPUT_DEV_CSV if split == "dev" else OUTPUT_TEST_CSV

    df = pd.read_csv(csv_file)
    y_true = df[[f"true_{c}" for c in LABEL_COLS]].values.astype(int)
    y_pred = df[[f"pred_{c}" for c in LABEL_COLS]].values.astype(int)

    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="micro", zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    exact_acc  = accuracy_score(y_true, y_pred)
    h_loss     = hamming_loss(y_true, y_pred)
    jacc_macro = jaccard_score(y_true, y_pred, average="macro", zero_division=0)
    jacc_samp  = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

    per_p, per_r, per_f1, per_sup = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    calibration = {}
    if "scores" in df.columns:
        for i, label in enumerate(LABEL_COLS):
            raw_scores = df["scores"].apply(
                lambda s: json.loads(s).get(label, 0) if isinstance(s, str) else 0
            ).values.astype(float)
            best_t, best_f1 = LABEL_THRESHOLDS[label], 0.0
            for t in range(2, 96, 2):
                preds_t = (raw_scores >= t).astype(int)
                _, _, f1, _ = precision_recall_fscore_support(
                    y_true[:, i], preds_t, average="binary", zero_division=0
                )
                if f1 > best_f1:
                    best_f1 = f1; best_t = t
            calibration[label] = (best_t, best_f1)

    skipped     = int(df["skipped_gpt"].eq(True).sum())  if "skipped_gpt"   in df.columns else 0
    gpt_calls   = len(df) - skipped
    consistency = int(df["used_consistency"].eq(True).sum()) if "used_consistency" in df.columns else 0
    embed_tok   = int(df["embed_tokens"].fillna(0).sum())    if "embed_tokens"   in df.columns else 0
    gpt_in_tok  = int(df["gpt_input_tokens"].fillna(0).sum()) if "gpt_input_tokens" in df.columns else 0
    gpt_out_tok = int(df["gpt_output_tokens"].fillna(0).sum()) if "gpt_output_tokens" in df.columns else 0
    mean_rt     = float(df["row_runtime_sec"].fillna(0).mean()) if "row_runtime_sec" in df.columns else 0.0
    cot_on      = bool(df["cot_enabled"].iloc[0])    if "cot_enabled"    in df.columns else None
    rules_on    = bool(df["heuristic_rules"].iloc[0]) if "heuristic_rules" in df.columns else None

    embed_cost = (embed_tok / 1000)  * EMBED_COST_PER_1K
    gin_cost   = (gpt_in_tok / 1000) * GPT_INPUT_COST_PER_1K
    gout_cost  = (gpt_out_tok / 1000)* GPT_OUTPUT_COST_PER_1K
    total_cost = embed_cost + gin_cost + gout_cost
    eval_elapsed = time.perf_counter() - eval_start

    W = 72
    print(f"\n{'=' * W}")
    print(f"   LABEL-AWARE RAG TOXICITY v5 — EVAL REPORT  [{split.upper()}]")
    print(f"{'=' * W}")
    print(f"  Generated   : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  GPT model   : {GPT_MODEL}   Embed: {EMBED_MODEL}")
    print(f"  Split       : {split}  ({'FINAL' if split=='test' else 'dev — diagnostic only'})")
    print(f"  CoT mode    : {'ON' if cot_on else 'OFF (JSON-only)'}")
    print(f"  Rules mode  : {'hybrid rule-augmented' if rules_on else 'pure model-driven'}")
    if split == "test" and os.path.exists(DEV_THRESHOLDS):
        print(f"  Thresholds  : dev-tuned (from {DEV_THRESHOLDS})")
    else:
        print(f"  Thresholds  : defaults")
    print(f"  Rows eval   : {len(df):,}")
    print()

    print("── GLOBAL METRICS ──")
    print(f"  Micro  Prec={micro_p:.4f}  Rec={micro_r:.4f}  F1={micro_f1:.4f}")
    print(f"  Macro  Prec={macro_p:.4f}  Rec={macro_r:.4f}  F1={macro_f1:.4f}")
    print(f"  Exact Match Acc  : {exact_acc:.4f}")
    print(f"  Hamming Loss     : {h_loss:.4f}")
    print(f"  Jaccard(macro)   : {jacc_macro:.4f}")
    print(f"  Jaccard(samples) : {jacc_samp:.4f}")
    print()

    print("── PER-LABEL METRICS ──\n")
    hdr = (f"  {'Label':<18}{'Prec':>8}{'Rec':>8}{'F1':>8}"
           f"{'TP':>7}{'FP':>7}{'FN':>7}{'TN':>7}{'Supp':>7}")
    if split == "dev":
        hdr += f"{'BestT':>7}{'BestF1':>8}"
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))

    for i, label in enumerate(LABEL_COLS):
        tn, fp, fn, tp = mcm[i].ravel()
        star = " ★" if label in PRECISION_PRIORITY else ""
        row = (f"  {label + star:<18}{per_p[i]:>8.4f}{per_r[i]:>8.4f}{per_f1[i]:>8.4f}"
               f"{tp:>7}{fp:>7}{fn:>7}{tn:>7}{int(per_sup[i]):>7}")
        if split == "dev":
            bt, bf = calibration.get(label, (LABEL_THRESHOLDS[label], 0.0))
            flag = " ◄" if bt != LABEL_THRESHOLDS[label] else ""
            row += f"{bt:>7}{bf:>8.4f}{flag}"
        print(row)

    if split == "test":
        print(f"\n{'!' * W}")
        print("  FINAL held-out test results. Thresholds tuned on dev, NOT this data.")
        print(f"{'!' * W}")
    print()

    print("── v5 FIXES ACTIVE ──")
    print(f"  [FIX 1] Severity rubric     : 4-point scale + dedicated severity check pass")
    print(f"  [FIX 2] Threat precision     : target-proximity regex + credibility filter")
    print(f"  [FIX 3] Identity recall      : group-noun boost + dedicated identity check pass")
    print(f"  [FIX 4] Skip-gate            : ultra-clean fast path (target 8-12% skip)")
    print(f"  [FIX 5] Heuristic logic      : R2 threshold=15, R4 severity floor, R5 IH→insult")
    print(f"  [FIX 6] Adaptive consistency : hard-labels only, weighted merge (0.6/0.4)")
    print(f"  [FIX 7] Prompt engineering   : boundary examples, failure-mode warnings")
    print(f"  [FIX 8] Threshold tuning     : recall floors + step=2 granularity")
    print(f"  Self-consistency  : {consistency:,} / {gpt_calls:,} GPT calls")
    print(f"  GPT skipped       : {skipped:,}")
    print()

    print("── COST & TIME ──")
    print(f"  Embed tokens    : {embed_tok:,}   cost ~${embed_cost:.6f}")
    print(f"  GPT input tok.  : {gpt_in_tok:,}   cost ~${gin_cost:.6f}")
    print(f"  GPT output tok. : {gpt_out_tok:,}   cost ~${gout_cost:.6f}")
    print(f"  Total cost est. : ${total_cost:.6f}")
    print(f"  Mean task time  : {mean_rt:.4f}s")
    print(f"  Eval compute    : {seconds_to_hms(eval_elapsed)}")
    print()
    print("✅ Evaluation complete.")
    print("=" * W)


# ============================================================
# RUN — WORKFLOW
# ============================================================

# ── Step 0: Split eval set (run once) ─────────────────────────
split_eval_set()

# ── Step 1: Build index ──────────────────────────────────────
build_index()

# ── Step 2: Classify dev set ─────────────────────────────────
await run_inference_async(split="dev")

# ── Step 3: Tune thresholds on dev ───────────────────────────
best_thresholds = tune_thresholds_on_dev()

# ── Step 4: Classify test set with tuned thresholds ──────────
best_thresholds = load_dev_thresholds()
await run_inference_async(split="test", thresholds=best_thresholds)

# ── Step 5: Final evaluation ─────────────────────────────────
evaluate_predictions("test")

SPLITTING EVAL SET (dev / held-out test)
Dev  rows : 3,999  → eval_dev_cl1.csv
Test rows : 4,001  → eval_test_cl1.csv

  Label                  Full%    Dev%   Test%
  toxic                  47.70   47.66   47.74
  severe_toxic           19.54   19.53   19.55
  obscene                35.69   35.66   35.72
  threat                  5.73    5.73    5.72
  insult                 34.62   34.61   34.64
  identity_hate          17.26   17.23   17.30
BUILD INDEX v5
Rows: 151,543
Label distribution:
toxic             11467
severe_toxic         32
obscene            5589
threat               20
insult             5102
identity_hate        24
is_clean         139330

  Pure clean: 136,772 (contaminated excluded: 2,558)
EMBEDDING
  [Batch   1/296]     512/151,543 (  0.34%) | tokens~  42,749 | cost~$0.000855
  [Batch   2/296]   1,024/151,543 (  0.68%) | tokens~  48,733 | cost~$0.000975
  [Batch   3/296]   1,536/151,543 (  1.01%) | tokens~  41,392 | cost~$0.000828
  [Batch   4/296]   2,048/151,543 